Implementation of cuttle [cubic illuminance](https://www.researchgate.net/profile/Christopher-Cuttle/publication/274538293_Research_Note_A_practical_approach_to_cubic_illuminance_measurement/links/56a6d84b08ae997e22ba4fc6/Research-Note-A-practical-approach-to-cubic-illuminance-measurement.pdf)

In [3]:
#write the cubic points from the points.txt file --> done
import numpy as np
# make grid points for cubic illumination
class CubicfromPoints:
    "this"
    def __init__(self, input_file, space_angle=0, output_file=None):
        self.input_file = input_file
        self.space_angle = space_angle
        self.output_file = output_file or input_file.replace('points.txt', f'points_cubic_spaceAngle{space_angle}.txt')
        
        self.cubic_dtype = np.dtype([
            ('point', np.float64, (3,)),       # (x, y, z)
            ('x_plus', np.float64, (3,)),      # x+
            ('x_minus', np.float64, (3,)),     # x-
            ('y_plus', np.float64, (3,)),      # y+
            ('y_minus', np.float64, (3,)),     # y-
            ('z_plus', np.float64, (3,)),      # z+
            ('z_minus', np.float64, (3,)),     # z-
            ('u_plus', np.float64, (3,)),      # u+
            ('u_minus', np.float64, (3,)),     # u-
            ('v_plus', np.float64, (3,)),      # v+
            ('v_minus', np.float64, (3,)),     # v-
            ('w_plus', np.float64, (3,)),      # w+
            ('w_minus', np.float64, (3,))      # w-
        ])
        
        self.rotation_matrix = self._compute_rotation_matrix()
        self.data_array = None
    def _compute_rotation_matrix(self):
        theta = -np.pi / 4
        phi = np.arctan(np.sqrt(2))

        R_y = np.array([
            [np.cos(theta), 0, np.sin(theta)],
            [0, 1, 0],
            [-np.sin(theta), 0, np.cos(theta)]
        ])
        
        R_x = np.array([
            [1, 0, 0],
            [0, np.cos(phi), -np.sin(phi)],
            [0, np.sin(phi), np.cos(phi)]
        ])
        
        return R_x @ R_y
    def _read_input(self):
        with open(self.input_file, 'r') as f:
            lines = f.readlines()
            data = []
            for line in lines:
                values = line.strip().split()
                x, y, z, *_ = map(float, values)
                
                x_plus = np.array([np.cos(self.space_angle), np.sin(self.space_angle), 0])
                x_minus = np.array([-np.cos(self.space_angle), -np.sin(self.space_angle), 0])
                y_plus = np.array([-np.sin(self.space_angle), np.cos(self.space_angle), 0])
                y_minus = np.array([np.sin(self.space_angle), -np.cos(self.space_angle), 0])
                z_plus = np.array([0, 0, 1])
                z_minus = np.array([0, 0, -1])
                
                u_plus = self.rotation_matrix @ x_plus
                u_minus = self.rotation_matrix @ x_minus
                v_plus = self.rotation_matrix @ y_plus
                v_minus = self.rotation_matrix @ y_minus
                w_plus = self.rotation_matrix @ z_plus
                w_minus = self.rotation_matrix @ z_minus
                
                data.append((np.array([x, y, z]), x_plus, x_minus, y_plus, y_minus, z_plus, z_minus,
                             u_plus, u_minus, v_plus, v_minus, w_plus, w_minus))
        
        self.data_array = np.array(data, dtype=self.cubic_dtype)
    def write_output(self):
        if self.data_array is None:
            self._read_input()
        
        with open(self.output_file, 'w') as f:
            for i in self.data_array:
                x = i['point'][0]
                y = i['point'][1]
                z = i['point'][2]
                f.write(f"{x} {y} {z} {i['u_plus'][0]} {i['u_plus'][1]} {i['u_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['u_minus'][0]} {i['u_minus'][1]} {i['u_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_plus'][0]} {i['v_plus'][1]} {i['v_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['v_minus'][0]} {i['v_minus'][1]} {i['v_minus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_plus'][0]} {i['w_plus'][1]} {i['w_plus'][2]}\n")
                f.write(f"{x} {y} {z} {i['w_minus'][0]} {i['w_minus'][1]} {i['w_minus'][2]}\n")
class cubic_illumination:
    def __init__(self, E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus):
        self.E_u_plus = E_u_plus
        self.E_u_minus = E_u_minus
        self.E_v_plus = E_v_plus
        self.E_v_minus = E_v_minus
        self.E_w_plus = E_w_plus
        self.E_w_minus = E_w_minus
        # outputs
        # illumination vector components
        E_u = self.E_u_plus - self.E_u_minus
        E_v = self.E_v_plus - self.E_v_minus
        E_w = self.E_w_plus - self.E_w_minus
        # illumination vector magnitudes
        abs_E = np.sqrt(E_u**2 + E_v**2 + E_w**2) # |E|
        # symmetric components
        m_E_u = (self.E_u_plus + self.E_u_minus - E_u)/2 # ~E_u
        m_E_v = (self.E_v_plus + self.E_v_minus - E_v)/2 # ~E_v
        m_E_w = (self.E_w_plus + self.E_w_minus - E_w)/2 # ~E_w
        # symmetric illuminance
        m_E = (m_E_u + m_E_v + m_E_w)/3 # ~E
        # scalar illuminance 
        Esr = m_E + abs_E/4 # Esr
        # unit vectors
        e_u = E_u/abs_E
        e_v = E_v/abs_E
        e_w = E_w/abs_E
        e_x = 0.707*(e_v - e_w)
        e_y = 0.816*e_u - 0.408*(e_v + e_w)
        e_z = 0.577*(e_u + e_v + e_w)
        # Horizontal working plane illuminance
        if e_z < 0:
            Ewp = m_E
        else:
            Ewp = abs_E*e_z + m_E
        # mean cylindrical illuminance
        Ecl = (abs_E*(np.sqrt(e_x**2 + e_y**2))/np.pi) + m_E
        
        self.E_u = E_u
        self.E_v = E_v
        self.E_w = E_w
        self.abs_E = abs_E
        self.m_E_u = m_E_u
        self.m_E_v = m_E_v
        self.m_E_w = m_E_w
        self.m_E = m_E
        self.Esr = Esr
        self.m_E = m_E
        self.Ewp = Ewp
        self.Ecl = Ecl

In [3]:
# make the cubic illumination points.txt 
import os
spaces = {"S01", "S02", "S04", "S09", "S07", "S08", "S09", "S10", "S11"}
# change current working directory to /scratch/DelftBlueFiles/Inputs
for space in spaces:
    input_file = rf"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/{space}/points.txt"
    # check if the file exists
    print(input_file)
    print(os.path.exists(input_file))
    CubicfromPoints(input_file, space_angle=0).write_output()

/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S02/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S10/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S04/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S08/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S07/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S09/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S11/points.txt
True
/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S01/points.txt
True


In [5]:
# example
E_u_plus = 10
E_u_minus = 230
E_v_plus = 320
E_v_minus = 50
E_w_plus = 40
E_w_minus = 530
CI = cubic_illumination(E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus)
print(CI.Ecl)

443.4042998137892


In [31]:
# calculate per-point cubic illumination from a tar.gz output file. 
import os
import numpy as np
import pandas as pd

result_path = os.path.join(r"../DelftBlueFiles/Outputs/s07_4_0_20240712_11_12_11_223547_d245c8b0-402e-11ef-85dd-4c5262490464.csv.gz")
grid_path = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S07/points_cubic_spaceAngle0.txt")

number_of_points_in_grid = 0
with open(grid_path, 'r') as f:
    for line in f:
        number_of_points_in_grid += 1
print("number of lines in grid file: ", number_of_points_in_grid)

number_of_points_in_result = 0
array = np.asarray(pd.read_csv(result_path, compression='gzip', header=None))
print("number of columns in result file: ", array.shape[1])

number of lines in grid file:  3714
number of columns in result file:  1416


In [9]:
import os
import numpy as np
import pandas as pd
# Debug, plot three numbers: number of lines in grid file, number of columns in result file, and number of lines in regenrated grid line

non_cubic_grid_oath = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S07/points.txt")
grid_path = os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S07/points_cubic_spaceAngle0.txt")
regenrated_grid_path = os.path.join(r"/scratch/nforouzandehsh/FormalizedLoD/regenerated_cubic.txt")
CubicfromPoints(os.path.join(r"/scratch/nforouzandehsh/DelftBlueFiles/Inputs/S07/points.txt"), space_angle=0,output_file=regenrated_grid_path).write_output()
result_path = os.path.join(r"../DelftBlueFiles/Outputs/s07_4_0_20240712_11_12_11_223547_d245c8b0-402e-11ef-85dd-4c5262490464.csv.gz")

number_of_points_in_grid_non_cubic = 0
with open(non_cubic_grid_oath, 'r') as f:
    for line in f:
        number_of_points_in_grid_non_cubic += 1
print("number of lines in non-cubic grid file*6: ", 6*number_of_points_in_grid_non_cubic)

number_of_points_in_grid = 0
with open(grid_path, 'r') as f:
    for line in f:
        number_of_points_in_grid += 1
print("number of lines in grid file: ", number_of_points_in_grid)

number_of_points_in_result = 0
array = np.asarray(pd.read_csv(result_path, compression='gzip', header=None))
print("number of columns in result file: ", array.shape[1])

number_of_points_in_regenrated_grid = 0
with open(regenrated_grid_path, 'r') as f:
    for line in f:
        number_of_points_in_regenrated_grid += 1
print("number of lines in regenrated grid file: ", number_of_points_in_regenrated_grid)

number of lines in non-cubic grid file*6:  3714
number of lines in grid file:  3714
number of columns in result file:  1416
number of lines in regenrated grid file:  3714


In [25]:
# print Grid Path content
class CubicIllumination_postprocess:
    """
    for each point in the grid, calculate different illuminance values and save them in a specific numpy dtype
    A class to calculate cubic illumination from a grid file. Final data type should be a numpy array with {point(x, y, z), E_u_plus, E_u_minus, E_v_plus, E_v_minus, E_w_plus, E_w_minus, illuminances values(COMPLETE THIS) ...} fields.

    Attributes:
        grid_path (str): The file path to the grid file.
        grid_type (str): The type of the grid file. Supported types are 'cubic' and 'planar'. 
            if 'planar', the grid will be generated, if 'cubic', the grid will be read directly from the .txt file.
        result_path (str): path to the simulated cubic csv.gz file.

    Methods:
        __init__(self, grid_path, grid_path_type='points.txt', result_path=None):
            Initializer. 
    """
    # data type for storing point information
    dtype = [
        ('centroid_coordinates', float, (3,)),  # x, y, z coordinates
        ('E_u_plus', float),
        ('E_u_minus', float),
        ('E_v_plus', float),
        ('E_v_minus', float),
        ('E_w_plus', float),
        ('E_w_minus', float),
        # Add more fields for illuminance values as needed
        ]
    def __init__(self, grid_path, grid_type='planar', result_path=None):
        # Pattern to match 'points.txt' or 'points_cubic_space{float}.txt' where {float} can be any float number
        if grid_type == 'planar':
            print('planar grid')
            # 1. make the cubic grid,
            # 2. repeat the code after 'cubic' grid part.
        elif grid_type == 'cubic':
            print('cubic grid')
            # 1. read the csv.gz file
            result = np.array(pd.read_csv(os.path.join(result_path)))
            print(result.shape)

            # with open(grid_path, 'r') as f:
            #     lines = f.readlines()
            #     for line in lines:
            #         print(line)
        else:
            raise ValueError(f"grid_path_type must be 'points.txt' or 'points_cubic_space{{float}}.txt' where {{float}} is any float number(space Angle)")
        # if the pattern matches the second, do y
# there are 6 fields in each line. line 0-5 have the same first three fields. this is the rule for the rest of the lines.
# the first fields are the x, y, z coordinates of the point. The rest of the three fields are:

In [13]:
cubic_object = CubicIllumination_postprocess(grid_path, grid_type='cubic', result_path=result_path)
# (8760: hours of the year, 1416:)

cubic grid
(8760, 1416)


In [26]:
number_of_lines = 0
with open(grid_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    number_of_lines += 1
print("number of unique points is" , number_of_lines/6)
# 1416

number of unique points is 619.0
